# Week 20: Knowledge Graphs from SEC Filings with Claude

Same logic as `build_company_graph.py`. Requires `LLM_API_KEY` and `LLM_MODEL` in a `.env` file (see `.env.example`) — this calls a real LLM (Anthropic's Messages API, via a direct `httpx` POST, no SDK) to extract entities and relationships. Also requires `SEC_USER_AGENT` in `.env`, and reuses `sec_thesis`'s SEC client and filings cache (Week 18) to fetch a real Apple 10-K.

`extract_relationships()` itself is provider-agnostic; only the `call_llm` function below is Anthropic-specific.

In [ ]:
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

from sec_thesis.cik import resolve_cik
from sec_thesis.config import load_settings
from sec_thesis.filing_parser import extract_text
from sec_thesis.filings import fetch_filings, list_filings
from sec_thesis.graph import (
    build_graph,
    competitors_of,
    most_central_entities,
    save_graph,
    visualize_graph,
)
from sec_thesis.llm.extraction import extract_relationships
from sec_thesis.sec_client import SECClient
from sec_thesis.storage.filings_db import FilingsDB

TICKER = "AAPL"
ANTHROPIC_MESSAGES_URL = "https://api.anthropic.com/v1/messages"

# A real 10-K's text can run to hundreds of thousands of characters; a
# production system would chunk and retrieve relevant sections (Weeks
# 9-11's RAG techniques). For this course exercise, truncate to a
# manageable prompt size instead.
MAX_FILING_TEXT_CHARS = 15_000


def call_llm(prompt: str) -> str:
    """The one Anthropic-specific piece; extract_relationships() itself is provider-agnostic."""
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            ANTHROPIC_MESSAGES_URL,
            headers={
                "x-api-key": os.environ["LLM_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": os.environ["LLM_MODEL"],
                "max_tokens": 4096,
                "messages": [{"role": "user", "content": prompt}],
            },
        )
        response.raise_for_status()
        data = response.json()
        for block in data["content"]:
            if block["type"] == "text":
                return block["text"]
        raise ValueError(f"No text block in response: {data}")


load_dotenv()
settings = load_settings()

## Fetch a Real Filing

Reuses Week 18's cached-fetch pipeline: resolve the CIK, list 10-Ks, download any not already cached.

In [ ]:
with SECClient(settings) as client, FilingsDB(settings.duckdb_path) as db:
    cik = resolve_cik(client, TICKER)
    list_filings(client, db, TICKER, cik, forms=["10-K"])
    fetch_filings(client, db, TICKER)
    filings = [f for f in db.query_by_ticker(TICKER) if f.local_path]

most_recent = sorted(filings, key=lambda f: f.filing_date, reverse=True)[0]
print(f"Using {TICKER} {most_recent.form} filed {most_recent.filing_date}")

html = Path(most_recent.local_path).read_text(encoding="utf-8", errors="ignore")
text = extract_text(html)[:MAX_FILING_TEXT_CHARS]
print(f"Extracted {len(text)} characters of filing text for the prompt.")

## Extract Entities and Relationships

In [ ]:
result = extract_relationships(TICKER, text, generate=call_llm)
print(f"Extracted {len(result.entities)} entities and {len(result.relationships)} relationships:")
for rel in result.relationships:
    print(f"  {rel.source} --[{rel.relation_type}]--> {rel.target}")
    print(f"    evidence: {rel.evidence!r}")

## Build and Query the Graph

In [ ]:
graph = build_graph([result])

print("Most central entities:")
for name, score in most_central_entities(graph):
    print(f"  {name}: {score:.3f}")

# The LLM chooses entity names freely (e.g. "Apple Inc." vs "Apple"), so
# look up the filer's own node by ticker rather than assuming a name.
filer = next((e for e in result.entities if e.ticker == TICKER), None)
if filer is not None:
    print(f"\nCompetitors of {filer.name}: {competitors_of(graph, filer.name)}")

## Visualize and Save

In [ ]:
graph_path = Path(settings.duckdb_path).parent / f"{TICKER}_graph.json"
save_graph(graph, graph_path)
print(f"Saved graph to {graph_path}")

image_path = graph_path.with_suffix(".png")
visualize_graph(graph, image_path)
print(f"Saved visualization to {image_path}")

## Display the Visualization

In [ ]:
from IPython.display import Image
Image(filename=str(image_path))